# KINNEWS - GPU experiments

Runs the BiLSTM experiments (E7-E9) and the Transformer fine-tuning (T1-T5).
Made for Kaggle: turn on a GPU and Internet in the session options, then use
**Save Version -> Save & Run All (Commit)** so it runs in the background (~2-2.5 h).
Everything ends up in the notebook's Output tab: `results/`, `results.zip` and `final_model/`.
Works on Colab too.

In [ ]:
import os
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
REPO_DIR = f"{WORK}/repo"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Samkwizera/NLP-application.git {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements-train.txt
!python scripts/prepare_data.py
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU!")

## BiLSTM (E7-E9)
- E7 random embeddings, max pooling
- E8 Word2Vec embeddings, max pooling
- E9 Word2Vec embeddings, attention pooling

In [ ]:
!python scripts/train_bilstm.py --run-name e7_bilstm_random_max --pooling max
!python scripts/train_bilstm.py --run-name e8_bilstm_w2v_max --pooling max --w2v
!python scripts/train_bilstm.py --run-name e9_bilstm_w2v_attention --pooling attention --w2v

## Transformers (T1-T3)
- T1 `xlm-roberta-base`, general multilingual
- T2 `Davlan/afro-xlmr-base`, XLM-R with extra pretraining on African languages incl. Kinyarwanda
- T3 `castorini/afriberta_base`, pretrained from scratch on African languages incl. Kinyarwanda

Checkpoints are deleted after each run, only the scores and predictions are kept, to save disk.

In [ ]:
!python scripts/train_transformer.py --model FacebookAI/xlm-roberta-base --run-name t1_xlmr_base
!rm -rf models/t1_xlmr_base
!python scripts/train_transformer.py --model Davlan/afro-xlmr-base --run-name t2_afroxlmr_base
!rm -rf models/t2_afroxlmr_base
!python scripts/train_transformer.py --model castorini/afriberta_base --run-name t3_afriberta_base
!rm -rf models/t3_afriberta_base

## Picking the best model on validation
The best of T1-T3 is chosen by **val** macro-F1 only. The test set is not used for any decision.

In [ ]:
import json
MODELS = {"t1_xlmr_base": "FacebookAI/xlm-roberta-base",
          "t2_afroxlmr_base": "Davlan/afro-xlmr-base",
          "t3_afriberta_base": "castorini/afriberta_base"}
val = {run: json.load(open(f"results/{run}.json"))["val"]["macro_f1"] for run in MODELS}
for run, score in val.items():
    print(f"{run:20s} val macro-F1 {score:.4f}")
BEST = MODELS[max(val, key=val.get)]
print("BEST =", BEST)

## T4 and T5
- T4: best model with 512 tokens instead of 256 (less of each article is cut off)
- T5: same as T4 plus balanced class weights in the loss, to help the rare topics

In [ ]:
!python scripts/train_transformer.py --model {BEST} --run-name t4_best_maxlen512 --max-length 512 --batch-size 8 --save-model
!rm -rf models/t4_best_maxlen512/checkpoint-*
!python scripts/train_transformer.py --model {BEST} --run-name t5_best_maxlen512_weighted --max-length 512 --batch-size 8 --class-weights --save-model
!rm -rf models/t5_best_maxlen512_weighted/checkpoint-*

## Summary and final model (picked on val)

In [ ]:
import glob, shutil
rows = [json.load(open(f)) for f in sorted(glob.glob("results/*.json"))]
for d in rows:
    print(f"{d['name']:42s} val mF1 {d['val']['macro_f1']:.4f}  test acc {d['test']['accuracy']:.4f}  test mF1 {d['test']['macro_f1']:.4f}")

final_run = max(["t4_best_maxlen512", "t5_best_maxlen512_weighted"],
                key=lambda r: json.load(open(f"results/{r}.json"))["val"]["macro_f1"])
print("final model:", final_run)
shutil.copytree(f"models/{final_run}/final", f"{WORK}/final_model", dirs_exist_ok=True)
shutil.make_archive(f"{WORK}/results", "zip", "results")

## Optional: upload the final model to the Hugging Face Hub
Only runs if a Kaggle secret called `HF_TOKEN` (a Write token) is attached under Add-ons -> Secrets.
Otherwise it is skipped and the model can be uploaded later from `final_model/`.

In [ ]:
HUB_ID = "Samkwizera/kinnews-topic-classifier"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
if token:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    AutoModelForSequenceClassification.from_pretrained(f"{WORK}/final_model").push_to_hub(HUB_ID, token=token)
    AutoTokenizer.from_pretrained(f"{WORK}/final_model").push_to_hub(HUB_ID, token=token)
    print("uploaded to https://huggingface.co/" + HUB_ID)
else:
    print("no HF_TOKEN, skipping upload")